# Fine-tuning SLM Comptable/Audit avec Unsloth

Ce notebook fine-tune **Llama 3.2 3B** sur des données comptables algériennes.

**Prérequis** : Activer le GPU dans Runtime > Change runtime type > T4 GPU

In [14]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
# Cellule 1 : Installation d'Unsloth
%%capture
!pip install unsloth
!pip install --upgrade pillow

UsageError: Line magic function `%%capture` not found.


In [ ]:
# Cellule 2 : Vérification du GPU
import torch
print(f"GPU disponible : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM : {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
else:
    raise RuntimeError("Aucun GPU détecté ! Activez le GPU dans Runtime > Change runtime type")

ModuleNotFoundError: No module named 'torch'

In [ ]:
# Cellule 3 : Chargement du modèle Llama 3.2 3B en 4-bit
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

print("Modèle chargé avec succès !")

In [ ]:
# Cellule 4 : Configuration des adapters LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=True,
    loftq_config=None,
)

print(f"Paramètres entraînables : {model.print_trainable_parameters()}")

In [ ]:
# Cellule 5 : Téléchargement du dataset
# Option A : Monter Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Copiez comptable_dataset_algerien.jsonl dans votre Drive avant de lancer
DATASET_PATH = "/content/drive/MyDrive/comptable_dataset_algerien.jsonl"

# Option B : Upload direct (décommentez si préféré)
# from google.colab import files
# uploaded = files.upload()  # Sélectionnez comptable_dataset_algerien.jsonl
# DATASET_PATH = "comptable_dataset_algerien.jsonl"

In [ ]:
# Cellule 6 : Préparation du dataset
from datasets import load_dataset

dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
print(f"Nombre d'exemples : {len(dataset)}")
print(f"Exemple : {dataset[0]}")

# Formatage en template Llama 3.1
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="llama-3.1",
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [
        tokenizer.apply_chat_template(
            convo, tokenize=False, add_generation_prompt=False
        )
        for convo in convos
    ]
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)
print(f"\nExemple formaté :\n{dataset[0]['text'][:500]}...")

In [ ]:
# Cellule 7 : Configuration de l'entraînement
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    dataset_num_proc=2,
    packing=True,
    args=TrainingArguments(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=150,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)

print("Entraînement configuré !")

In [ ]:
# Cellule 8 : Lancement de l'entraînement
print("Début de l'entraînement...")
trainer_stats = trainer.train()
print(f"\nEntraînement terminé !")
print(f"Durée : {trainer_stats.metrics['train_runtime']:.0f} secondes")
print(f"Loss finale : {trainer_stats.metrics['train_loss']:.4f}")

In [ ]:
# Cellule 9 : Test du modèle fine-tuné
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": "Tu es un expert-comptable et auditeur algérien. Tu utilises le Plan Comptable National (PCN) algérien."},
    {"role": "user", "content": "Quelle est la différence entre SARL et EURL en Algérie ?"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=512,
    use_cache=True,
    temperature=0.3,
)

response = tokenizer.batch_decode(outputs)
print(response[0])

In [ ]:
# Cellule 10 : Export en GGUF pour Ollama
model.save_pretrained_gguf(
    "comptable-slm",
    tokenizer,
    quantization_method="q4_k_m",
)

print("\nExport GGUF terminé !")
print("Le fichier .gguf sera dans le dossier comptable-slm/")

In [ ]:
# Cellule 11 : Téléchargement du fichier GGUF
import glob
import os
from google.colab import files

gguf_files = glob.glob("comptable-slm/*.gguf")
if gguf_files:
    print(f"Fichiers GGUF trouvés : {gguf_files}")
    for f in gguf_files:
        size_mb = os.path.getsize(f) / 1024 / 1024
        print(f"  {f} ({size_mb:.0f} MB)")
        files.download(f)
else:
    print("Aucun fichier GGUF trouvé. Vérifiez l'étape 10.")